# Lab 7 - Facial Recognition

### Install the required libraries

In [1]:
%pip install opencv-contrib-python==4.10.0.84 deepface werkzeug numpy

Note: you may need to restart the kernel to use updated packages.


### Import the libraries
- **cv2:** Access the webcam and process video frames.
- **deepface:** Perform face analysis and recognition using deep learning models.
- **numpy:** Perform numerical calculations on arrays.
- **werkzeug:** Provide utility functions used by Python web applications and other libraries. (deepface environment required this)

In [2]:
import cv2
import csv
import time
import numpy as np

from pathlib import Path
from datetime import datetime
from deepface import DeepFace

### Configuration

In [3]:
BASE_DIR = Path.cwd()
DATABASE_DIR = BASE_DIR / "faces_db"
CSV_PATH = BASE_DIR / "attendance.csv"

MODEL_NAME = "Facenet512"
DETECTOR_BACKEND = "retinaface"

RECOGNITION_THRESHOLD = 0.30 # Lower than threshold, correct, else, false
EXIT_DELAY = 5.0 # Must remain undetected 5 secs before recognized as EXIT

### Supporting Functions

In [4]:
# Cos distance
# Cos distance = 1 - Cos similarity
def cosine_distance(embedding1, embedding2):

    embedding1 = np.asarray(embedding1, dtype=np.float32)
    embedding2 = np.asarray(embedding2, dtype=np.float32)

    norm1 = np.linalg.norm(embedding1)
    norm2 = np.linalg.norm(embedding2)

    if norm1 == 0 or norm2 == 0:
        return 1.0

    similarity = np.dot(embedding1, embedding2) / (norm1 * norm2)

    return float(1.0 - similarity)

# Load and extract embeddings
def load_face_database():

    database = {}

    person_folders = [
        folder for folder in DATABASE_DIR.iterdir()
        if folder.is_dir()
    ]

    for person_folder in person_folders:
        person_name = person_folder.name
        database[person_name] = []

        image_paths = [
            path for path in person_folder.iterdir()
            if path.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]
        ]

        if not image_paths:
            print(f"No images found for {person_name}")
            continue

        for image_path in image_paths:
            try:
                embedding_result = DeepFace.represent(
                    img_path=str(image_path),
                    model_name=MODEL_NAME,
                    detector_backend=DETECTOR_BACKEND,
                    enforce_detection=True
                )

                if embedding_result:
                    embedding = embedding_result[0]["embedding"]

                    database[person_name].append(embedding)

                    print(f"Loaded {image_path.name} for {person_name}")

            except Exception as error:
                print(
                    f"Could not process {image_path.name}: "
                    f"{error}"
                )

        if not database[person_name]:
            print(
                f"No usable face embeddings for {person_name}"
            )

    # Remove people for whom no usable embeddings were generated.
    database = {
        name: embeddings
        for name, embeddings in database.items()
        if embeddings
    }

    if not database:
        raise ValueError(
            "No valid face embeddings could be loaded. "
            "Check the images inside faces_db."
        )

    for person_name, embeddings in database.items():
        print(f"  {person_name}: {len(embeddings)} reference image(s)")

    return database

# Recognizing (Compare embeddings)
def recognize_face(face_embedding, database):

    best_name = "Unknown"
    best_distance = float("inf")

    for person_name, reference_embeddings in database.items():
        for reference_embedding in reference_embeddings:

            distance = cosine_distance(
                face_embedding,
                reference_embedding
            )

            if distance < best_distance:
                best_distance = distance
                best_name = person_name

    if best_distance >= RECOGNITION_THRESHOLD:
        best_name = "Unknown"

    return best_name, best_distance

# Store event (ENTER / EXIT)
def log_event(person_name, event):

    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    with open(
        CSV_PATH,
        mode="a",
        newline="",
        encoding="utf-8"
    ) as file:

        writer = csv.writer(file)

        writer.writerow([
            person_name,
            event,
            timestamp
        ])

    print(f"{timestamp} | {person_name} | {event}")


def initialize_attendance_csv():

    if not CSV_PATH.exists():
        with open(
            CSV_PATH,
            mode="w",
            newline="",
            encoding="utf-8"
        ) as file:

            writer = csv.writer(file)

            writer.writerow([
                "name",
                "event",
                "timestamp"
            ])

### Start the system
- Load the database
- Initialize the webcam
- Continuously detecting faces
- Take attendance

In [5]:
# Load database and attendance csv

database = load_face_database()

initialize_attendance_csv()

attendance_state = {}

for person_name in database:
    attendance_state[person_name] = {
        "present": False,
        "last_seen": 0.0
    }


# Initialize Webcam

camera = cv2.VideoCapture(0)

if not camera.isOpened():
    raise RuntimeError(
        "Could not open webcam. Check your camera connection."
    )

print("Webcam started, press Q to quit\n")


# Looping
try:
    while True:

        success, frame = camera.read()

        if not success:
            print("[WARNING] Could not read webcam frame.")
            break

        # Store identities recognized in this frame.
        recognized_people = set()

        # Each frame may contain multiple faces.
        try:
            face_results = DeepFace.represent(
                img_path=frame,
                model_name=MODEL_NAME,
                detector_backend=DETECTOR_BACKEND,
                enforce_detection=False,
                align=True
            )

            for face_result in face_results:

                # Skip entries without a detected face region.
                facial_area = face_result.get("facial_area")

                if not facial_area:
                    continue

                x = int(facial_area["x"])
                y = int(facial_area["y"])
                w = int(facial_area["w"])
                h = int(facial_area["h"])

                # Ensure coordinates stay within the image.
                x = max(0, x)
                y = max(0, y)

                x2 = min(frame.shape[1], x + w)
                y2 = min(frame.shape[0], y + h)

                if x2 <= x or y2 <= y:
                    continue

                face_embedding = face_result["embedding"]

                best_name, best_distance = recognize_face(
                    face_embedding,
                    database
                )

                # A recognized identity counts as present.
                if best_name != "Unknown":
                    recognized_people.add(best_name)
                    box_color = (0, 255, 0)
                else:
                    box_color = (0, 0, 255)

                # Draw the bounding box.
                cv2.rectangle(
                    frame,
                    (x, y),
                    (x2, y2),
                    box_color,
                    2
                )

                # Display the name and distance.
                if best_name == "Unknown":
                    label = f"Unknown ({best_distance:.2f})"
                else:
                    label = f"{best_name} ({best_distance:.2f})"

                label_y = max(25, y - 10)

                cv2.putText(
                    frame,
                    label,
                    (x, label_y),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.65,
                    box_color,
                    2
                )

        # Keep the webcam running if face processing failson a particular frame.
        except Exception as error:
            print(f"[WARNING] Face processing failed: {error}")



        # Update attendance status
        current_time = time.time()

        for person_name, state in attendance_state.items():

            if person_name in recognized_people:

                # Update the last time this person was detected.
                state["last_seen"] = current_time

                # Log ENTER only once per attendance session.
                if not state["present"]:
                    state["present"] = True

                    log_event(
                        person_name,
                        "ENTER"
                    )

            else:
                # If they were present, check how long they have been missing from the camera.
                if state["present"]:

                    absent_duration = (
                        current_time - state["last_seen"]
                    )

                    if absent_duration >= EXIT_DELAY:

                        state["present"] = False

                        log_event(
                            person_name,
                            "EXIT"
                        )

        # Display attendance information
        present_names = [
            name
            for name, state in attendance_state.items()
            if state["present"]
        ]

        cv2.putText(
            frame,
            f"Present: {len(present_names)}",
            (10, 30),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (255, 255, 255),
            2
        )

        cv2.imshow(
            "Facial Recognition Attendance",
            frame
        )

        # Press Q to quit.
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

finally:
    # Release webcam and close windows even if an error occurs.
    camera.release()
    cv2.destroyAllWindows()

    print("\nWebcam stopped")

Loaded 45_degree_left_glass.jpg for bach
Loaded 45_degree_left_no_glass.jpg for bach
Loaded 45_degree_right_glass.jpg for bach
Loaded 45_degree_right_no_glass.jpg for bach
Loaded front_glass.jpg for bach
Loaded front_no_glass.jpg for bach
Loaded left_glass.jpg for bach
Loaded left_no_glass.jpg for bach
Loaded right_glass.jpg for bach
Loaded right_no_glass.jpg for bach
Loaded slightly_left_glass.jpg for bach
Loaded slightly_left_no_glass.jpg for bach
Loaded slightly_right_glass.jpg for bach
Loaded slightly_right_no_glass.jpg for bach
Loaded huy_down.jpg for huy
Loaded huy_front.jpg for huy
Loaded huy_side.jpg for huy
Loaded angry.jpg for manh
Loaded down.jpg for manh
Loaded front.jpg for manh
Loaded laugh.jpg for manh
Loaded left.jpg for manh
Loaded right.jpg for manh
Loaded sad.jpg for manh
Loaded smile.jpg for manh
Loaded up.jpg for manh
Loaded WIN_20261028_09_02_33_Pro.jpg for manh
Loaded WIN_20261028_09_02_37_Pro.jpg for manh
Loaded WIN_20261028_09_02_40_Pro.jpg for manh
Loaded WIN_